In [1]:
import nltk
from nltk.stem import WordNetLemmatizer,PorterStemmer
from nltk.corpus import wordnet,stopwords
import spacy
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
nlp = spacy.load('en_core_web_sm')
lemmatizer = WordNetLemmatizer()

In [2]:
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # Default

words = ["studies","am","went","feet","flying","cars","mice","better","ate"]
pos_tags = nltk.pos_tag(words)

for word, tag in pos_tags:
    lemma = lemmatizer.lemmatize(word, get_wordnet_pos(tag))
    print(f"{word} ({tag}) → {lemma}")

studies (NNS) → study
am (VBP) → be
went (VBD) → go
feet (NNS) → foot
flying (VBG) → fly
cars (NNS) → car
mice (RB) → mice
better (RB) → well
ate (VB) → eat


In [3]:
def lemmatize_sentence_list(sent_list):
    results = []
    for sent in sent_list:
        doc = nlp(sent)
        lemmas = " ".join([token.lemma_ for token in doc])
        results.append((sent, lemmas))
    return results

sentences = ["The students are studying hard.", "He went to the market."]
for orig, lem in lemmatize_sentence_list(sentences):
    print(f"Original: {orig}\nLemmatized: {lem}\n")

Original: The students are studying hard.
Lemmatized: the student be study hard .

Original: He went to the market.
Lemmatized: he go to the market .



In [4]:
stemmer = PorterStemmer()
words = ["studies","better","flying","easily","running"]

print(f"{'Word':<10}{'Stem':<15}{'Lemma':<15}")
for w in words:
    stem = stemmer.stem(w)
    lemma = lemmatizer.lemmatize(w, get_wordnet_pos(nltk.pos_tag([w])[0][1]))
    print(f"{w:<10}{stem:<15}{lemma:<15}")


Word      Stem           Lemma          
studies   studi          study          
better    better         well           
flying    fli            fly            
easily    easili         easily         
running   run            run            


In [5]:
para = """My university is located in a beautiful city with modern buildings, libraries, and research centers. 
Students from all over the country come here to study and innovate."""

orig, lemmatized = lemmatize_sentence_list([para])[0]
words = [w for w in lemmatized.split()]
total_words = len(words)
unique_lemmas = len(set(words))
print(f"Total Words: {total_words}\nUnique Lemmas: {unique_lemmas}")

Total Words: 31
Unique Lemmas: 28


In [6]:
text = "The children are running and playing happily."
pos_tags = nltk.pos_tag(nltk.word_tokenize(text))
for word, tag in pos_tags:
    lemma = lemmatizer.lemmatize(word, get_wordnet_pos(tag))
    print(f"{word} ({tag}) → {lemma}")

The (DT) → The
children (NNS) → child
are (VBP) → be
running (VBG) → run
and (CC) → and
playing (VBG) → play
happily (RB) → happily
. (.) → .


In [8]:
stop_words = set(stopwords.words('english'))
text = "The students are studying hard for their final examinations."
doc = nlp(text)

clean_tokens = [token.lemma_ for token in doc if token.text.lower() not in stop_words and token.is_alpha]
print("Clean Tokens:", clean_tokens)
print("Remaining Count:", len(clean_tokens))

Clean Tokens: ['student', 'study', 'hard', 'final', 'examination']
Remaining Count: 5


In [9]:
sentence = "Dogs are chasing the cats who were running through the gardens."

# spaCy
doc = nlp(sentence)
spacy_lemmas = [t.lemma_ for t in doc]

# NLTK
tokens = nltk.word_tokenize(sentence)
pos_tags = nltk.pos_tag(tokens)
nltk_lemmas = [lemmatizer.lemmatize(w, get_wordnet_pos(p)) for w, p in pos_tags]

print("spaCy Lemmas:", spacy_lemmas)
print("NLTK Lemmas:", nltk_lemmas)

spaCy Lemmas: ['dog', 'be', 'chase', 'the', 'cat', 'who', 'be', 'run', 'through', 'the', 'garden', '.']
NLTK Lemmas: ['Dogs', 'be', 'chase', 'the', 'cat', 'who', 'be', 'run', 'through', 'the', 'garden', '.']


In [10]:
def clean_text(text):
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop]
    return tokens

print(clean_text("Running runners run faster when the weather is better!"))


['run', 'runner', 'run', 'fast', 'weather', 'well']


In [11]:
paragraph = """Karachi is a large city. The people of Karachi love to work hard and enjoy the sea view. 
Many people visit Karachi every year."""

lemmas = clean_text(paragraph)
lemma_counts = Counter(lemmas)
print(lemma_counts.most_common(5))

[('karachi', 3), ('people', 2), ('large', 1), ('city', 1), ('love', 1)]


In [12]:
present = "Students write their reports and attend lectures."
past = "Students wrote their reports and attended lectures."

for label, text in [("Present", present), ("Past", past)]:
    lemmas = [t.lemma_ for t in nlp(text) if t.pos_ == "VERB"]
    print(f"{label} tense verbs → {lemmas}")

Present tense verbs → ['write', 'attend']
Past tense verbs → ['write', 'attend']


In [13]:
data = {'Review': ['The product was amazing and worked perfectly.', 'I am loving this great experience!']}
df = pd.DataFrame(data)

df['Clean_Review'] = df['Review'].apply(lambda x: " ".join(clean_text(x)))
print(df)

                                          Review  \
0  The product was amazing and worked perfectly.   
1             I am loving this great experience!   

                     Clean_Review  
0  product amazing work perfectly  
1           love great experience  


In [14]:
class MyLemmatizer:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")

    def lemmatize(self, text):
        doc = self.nlp(text)
        return [t.lemma_ for t in doc if t.is_alpha]

    def compare(self, s1, s2):
        lemmas1, lemmas2 = set(self.lemmatize(s1)), set(self.lemmatize(s2))
        overlap = len(lemmas1 & lemmas2) / len(lemmas1 | lemmas2) * 100
        print(f"Lemma Overlap: {overlap:.2f}%")

my_lem = MyLemmatizer()
my_lem.compare("Students are learning AI.", "Students learned AI in class.")


Lemma Overlap: 50.00%
